# Probabilistic PCA, EM-PCA, Bayesian PCA, Factor Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/probabilistic_pca_variants.ipynb)

Companion notebook to the [blog post](https://sesen.ai/blog/probabilistic-pca-em-bayesian-factor-analysis). Four flavours of PCA in plain NumPy, all built on the same probabilistic latent variable model:

1. **ML-PCA closed form** — the eigendecomposition solution of Tipping & Bishop (1999b).
2. **EM-PCA** — iterative E/M updates, scales to high $D$, handles missing data.
3. **Bayesian PCA (ARD)** — picks the latent dimensionality automatically.
4. **Factor Analysis** — diagonal noise covariance instead of isotropic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## 1. ML-PCA closed form

$\mathbf{W}_{ML} = \mathbf{U}_M (\boldsymbol{\Lambda}_M - \sigma^2 \mathbf{I})^{1/2}\mathbf{R}$, where $\sigma^2 = (D-M)^{-1}\sum_{i=M+1}^D \lambda_i$.

In [ ]:
def ppca_closed_form(X, M):
    N, D = X.shape
    mu = X.mean(axis=0)
    S = ((X - mu).T @ (X - mu)) / N
    eigvals, eigvecs = np.linalg.eigh(S)
    eigvals, eigvecs = eigvals[::-1], eigvecs[:, ::-1]
    sigma2 = float(np.mean(eigvals[M:])) if M < D else 0.0
    W = eigvecs[:, :M] @ np.diag(np.sqrt(np.maximum(eigvals[:M] - sigma2, 0)))
    return mu, W, sigma2

### Quick win: tilted 2D blob

In [ ]:
theta = np.deg2rad(20.0)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
X = (rng.standard_normal((120, 2)) * np.array([1.6, 0.35])) @ R.T

mu, W, sigma2 = ppca_closed_form(X, M=1)
print(f'W = {W.ravel()},  sigma^2 = {sigma2:.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[:, 0], X[:, 1], s=14, alpha=0.5, color='#2d6cdf')
w = W.flatten()
ax.plot([mu[0]-3*w[0], mu[0]+3*w[0]], [mu[1]-3*w[1], mu[1]+3*w[1]], color='#cc3344', lw=2.5)
ax.scatter([mu[0]], [mu[1]], color='black', s=50, marker='x')
ax.set_title(f'Closed-form PPCA  $\\sigma^2$={sigma2:.4f}')
ax.set_aspect('equal', 'box')
ax.grid(alpha=0.25)
plt.show()

## 2. EM-PCA

Iterative algorithm with $O(NDM)$ cost per iteration. Bishop equations 12.54-12.57.

In [ ]:
def ppca_em(X, M, n_iter=80, seed=0):
    rng = np.random.default_rng(seed)
    N, D = X.shape
    mu = X.mean(axis=0)
    Xc = X - mu
    W = rng.standard_normal((D, M)) * 0.1
    sigma2 = 1.0
    for _ in range(n_iter):
        M_mat = W.T @ W + sigma2 * np.eye(M)
        M_inv = np.linalg.inv(M_mat)
        Ez = Xc @ W @ M_inv
        Ezz_sum = N * sigma2 * M_inv + Ez.T @ Ez
        W = (Xc.T @ Ez) @ np.linalg.inv(Ezz_sum)
        sigma2 = max((np.sum(Xc**2) - 2*np.sum((Ez @ W.T) * Xc)
                      + np.trace(Ezz_sum @ (W.T @ W))) / (N * D), 1e-10)
    return mu, W, sigma2

mu_cf, W_cf, s2_cf = ppca_closed_form(X, M=1)
mu_em, W_em, s2_em = ppca_em(X, M=1, n_iter=80, seed=42)

cos_sim = abs(float(np.dot(W_cf.flatten() / np.linalg.norm(W_cf),
                            W_em.flatten() / np.linalg.norm(W_em))))
print(f'EM vs closed-form: cos(W_cf, W_em) = {cos_sim:.6f}')
print(f'sigma^2 closed-form = {s2_cf:.6f},  EM = {s2_em:.6f}')

Cosine similarity 1.0 to six decimal places, and the two $\sigma^2$ estimates agree to better than $10^{-5}$. EM and the closed form converge to the same subspace, as they must.

## 3. Bayesian PCA via ARD

Place an independent Gaussian prior on each column $\mathbf{w}_i$ of $\mathbf{W}$ with its own precision $\alpha_i$. Surplus columns get driven to zero. Update rule: $\alpha_i^{new} = D / \|\mathbf{w}_i\|^2$.

In [ ]:
def bayesian_pca_ard(X, M, n_iter=200, seed=0, eps=1e-8):
    rng = np.random.default_rng(seed)
    N, D = X.shape
    mu = X.mean(axis=0)
    Xc = X - mu
    W = rng.standard_normal((D, M)) * 0.1
    sigma2 = 1.0
    alpha = np.ones(M)
    for _ in range(n_iter):
        M_mat = W.T @ W + sigma2 * np.eye(M)
        M_inv = np.linalg.inv(M_mat)
        Ez = Xc @ W @ M_inv
        Ezz_sum = N * sigma2 * M_inv + Ez.T @ Ez
        W = (Xc.T @ Ez) @ np.linalg.inv(Ezz_sum + sigma2 * np.diag(alpha))
        sigma2 = max((np.sum(Xc**2) - 2*np.sum((Ez @ W.T) * Xc)
                      + np.trace(Ezz_sum @ (W.T @ W))) / (N * D), 1e-10)
        alpha = D / (np.sum(W**2, axis=0) + eps)
    return mu, W, sigma2, alpha

# D=10 data with true rank M_true=3 (high-variance) + 7 noise dirs
rng2 = np.random.default_rng(1)
D, M_true, N = 10, 3, 300
Z = rng2.standard_normal((N, D))
sds = np.array([1.0]*M_true + [0.5]*(D - M_true))
X10 = Z * sds
Q, _ = np.linalg.qr(rng2.standard_normal((D, D)))
X10 = X10 @ Q.T

# Over-specify M = 9; ARD should prune 6 columns
_, W_b, s2_b, alpha_b = bayesian_pca_ard(X10, M=9, n_iter=300, seed=0)
col_norms = np.linalg.norm(W_b, axis=0)
print(f'Column norms: {col_norms.round(3)}')
print(f'Columns with norm > 0.05: {int(np.sum(col_norms > 0.05))} '
      f'(true rank = {M_true})')

In [ ]:
# Compare column norms: ML-PPCA vs Bayesian PCA
_, W_ml, _ = ppca_closed_form(X10, M=9)
fig, ax = plt.subplots(figsize=(8, 4.5))
width = 0.4
x = np.arange(9)
ax.bar(x - width/2, np.linalg.norm(W_ml, axis=0), width, label='ML-PPCA', color='#888888')
ax.bar(x + width/2, col_norms, width, label='Bayesian PCA (ARD)', color='#2d6cdf')
ax.set_xlabel('latent column index')
ax.set_ylabel(r'$\|\mathbf{w}_i\|$')
ax.set_title('ARD drives surplus columns to zero')
ax.legend()
ax.grid(alpha=0.25, axis='y')
plt.show()

## 4. Factor Analysis

Same model as PPCA but with diagonal noise covariance $\boldsymbol{\Psi}$ instead of $\sigma^2 \mathbf{I}$. Right choice when features have very different noise floors.

In [ ]:
def factor_analysis_em(X, M, n_iter=200, seed=0):
    rng = np.random.default_rng(seed)
    N, D = X.shape
    mu = X.mean(axis=0)
    Xc = X - mu
    W = rng.standard_normal((D, M)) * 0.1
    Psi = np.var(Xc, axis=0) + 1e-3
    for _ in range(n_iter):
        Psi_inv_W = W / Psi[:, None]
        G = np.linalg.inv(np.eye(M) + W.T @ Psi_inv_W)
        Ez = Xc @ Psi_inv_W @ G
        Ezz_sum = N * G + Ez.T @ Ez
        W = (Xc.T @ Ez) @ np.linalg.inv(Ezz_sum)
        cross = np.sum(W * ((Xc.T @ Ez) / N), axis=1)
        Psi = np.maximum(np.mean(Xc**2, axis=0) - cross, 1e-6)
    return mu, W, Psi

def gauss_ll(X, mu, Cov):
    D = X.shape[1]
    sign, logdet = np.linalg.slogdet(Cov)
    Cinv = np.linalg.inv(Cov)
    Xc = X - mu
    quad = np.einsum('ni,ij,nj->n', Xc, Cinv, Xc)
    return float(np.sum(-0.5*(D*np.log(2*np.pi) + logdet + quad)))

# Heteroscedastic data: 2 true factors + per-feature noise of very different scales
rng3 = np.random.default_rng(2)
D_h, M_h, N_h = 6, 2, 800
W_true = rng3.standard_normal((D_h, M_h)) * 0.8
Z_h = rng3.standard_normal((N_h, M_h))
noise_sd = np.array([0.05, 0.05, 0.05, 1.5, 1.5, 1.5])
X_h = Z_h @ W_true.T + rng3.standard_normal((N_h, D_h)) * noise_sd

# 75/25 train/test
idx = np.random.default_rng(20).permutation(N_h)
X_tr, X_te = X_h[idx[:600]], X_h[idx[600:]]

mu_p, W_p, s2_p = ppca_closed_form(X_tr, M=M_h)
C_p = W_p @ W_p.T + s2_p * np.eye(D_h)
ll_p = gauss_ll(X_te, mu_p, C_p) / X_te.shape[0]

mu_f, W_f, Psi_f = factor_analysis_em(X_tr, M=M_h, n_iter=300, seed=3)
C_f = W_f @ W_f.T + np.diag(Psi_f)
ll_f = gauss_ll(X_te, mu_f, C_f) / X_te.shape[0]

print(f'Held-out log-likelihood per sample:')
print(f'  PPCA: {ll_p:.3f}')
print(f'  FA:   {ll_f:.3f}')
print(f'  FA gain: +{ll_f - ll_p:.3f} nats/sample')
print(f'PPCA sigma^2 = {s2_p:.3f}')
print(f'True noise variances: {(noise_sd**2).round(3)}')
print(f'FA-learned Psi:       {Psi_f.round(3)}')

## 5. EM-PCA with missing data

Treat missing entries as extra latents. The E step infers them; the M step uses the imputed values.

In [ ]:
def ppca_em_missing(X_obs, mask, M, n_iter=150, seed=0):
    rng = np.random.default_rng(seed)
    N, D = X_obs.shape
    col_mean = np.array([X_obs[mask[:, j], j].mean() for j in range(D)])
    X = X_obs.copy()
    for j in range(D):
        X[~mask[:, j], j] = col_mean[j]
    mu = X.mean(axis=0)
    W = rng.standard_normal((D, M)) * 0.1
    sigma2 = 1.0
    for _ in range(n_iter):
        Xc = X - mu
        M_mat = W.T @ W + sigma2 * np.eye(M)
        M_inv = np.linalg.inv(M_mat)
        Ez = Xc @ W @ M_inv
        Ezz_sum = N * sigma2 * M_inv + Ez.T @ Ez
        X_pred = Ez @ W.T + mu
        X = np.where(mask, X_obs, X_pred)
        mu = X.mean(axis=0)
        Xc = X - mu
        W = (Xc.T @ Ez) @ np.linalg.inv(Ezz_sum)
        sigma2 = max((np.sum(Xc**2) - 2*np.sum((Ez @ W.T) * Xc)
                      + np.trace(Ezz_sum @ (W.T @ W))) / (N * D), 1e-10)
    return mu, W, sigma2, X

rng4 = np.random.default_rng(11)
n_m = 200
z = rng4.standard_normal((n_m, 2)) * np.array([1.5, 0.6])
W_true_m = np.array([[1.0, 0.0], [0.6, 0.8], [-0.3, 0.5]])
X_full = z @ W_true_m.T + rng4.standard_normal((n_m, 3)) * 0.15
mask = rng4.random((n_m, 3)) > 0.30

_, _, _, X_imp = ppca_em_missing(X_full, mask, M=2, n_iter=150, seed=0)
missing_rmse = np.sqrt(np.mean((X_imp[~mask] - X_full[~mask])**2))
print(f'Missing-entry RMSE: {missing_rmse:.4f}')
print(f'Fraction missing: {(~mask).mean():.2%}')

## Exercises

1. **Eigenvalue limit**: Take the EM-PCA function and set `sigma2 = 1e-12` (don't update it). Show that the columns of $\mathbf{W}$ at convergence are scaled versions of the top-$M$ eigenvectors of the data covariance (this is Roweis's $\sigma^2 \to 0$ limit recovering standard PCA).

2. **MNIST-style heteroscedastic data**: Load `sklearn.datasets.load_digits()` and corrupt half the pixels by adding $\mathcal{N}(0, 4)$ noise (keep the other half clean). Compare PPCA, FA, and `sklearn.decomposition.FactorAnalysis` on held-out log-likelihood.

3. **ARD with `M=D`**: Run `bayesian_pca_ard` with `M=D` on the 10D data. How many columns survive? Try a smaller `N` (50 instead of 300). Does ARD still find the right rank?

4. **Subspace rotation**: PPCA's $\mathbf{W}_{ML}$ is defined only up to an $M \times M$ rotation $\mathbf{R}$. Show numerically that two independent EM runs with different seeds produce $\mathbf{W}_1$ and $\mathbf{W}_2$ that span the same subspace (use $\mathbf{W}_1 (\mathbf{W}_1^T \mathbf{W}_1)^{-1} \mathbf{W}_1^T = \mathbf{W}_2 (\mathbf{W}_2^T \mathbf{W}_2)^{-1} \mathbf{W}_2^T$ as the projector test).